# google_cloud_logging smoke test

Smoke test for `google-cloud-logging`, the client for Google Cloud Logging. Every example runs
offline: it builds a client, loggers, log entries, a batch, a sink and a metric with anonymous
credentials, and formats records with the structured-log handler. No request is sent to Cloud Logging.

## Example 1: Client and Logger construction

In [ ]:
# Example 1: Client and Logger construction. AnonymousCredentials keep this offline: no API call is made
# until a method that talks to the service runs.
import google.cloud.logging
from google.auth.credentials import AnonymousCredentials

client = google.cloud.logging.Client(project="demo-project", credentials=AnonymousCredentials())
print(f"client: {type(client).__name__}, project: {client.project}")
assert client.project == "demo-project", client.project

logger = client.logger("demo", labels={"env": "test"})
print(f"logger name: {logger.name}")
print(f"logger full_name: {logger.full_name}")
print(f"logger path: {logger.path}")
assert logger.name == "demo", logger.name
assert logger.full_name == "projects/demo-project/logs/demo", logger.full_name
assert logger.path == "/projects/demo-project/logs/demo", logger.path
assert logger.labels == {"env": "test"}, logger.labels
assert google.cloud.logging.logger_name_from_path(logger.full_name) == "demo"

## Example 2: Log entries and their API representation

In [ ]:
# Example 2: Log entries and their API representation (the JSON the service receives).
from google.cloud.logging import Resource, StructEntry, TextEntry

resource = Resource(type="global", labels={"project_id": "demo-project"})
text_entry = TextEntry(
    log_name="projects/demo-project/logs/demo",
    payload="hello from the smoke test",
    severity="WARNING",
    labels={"env": "test"},
    resource=resource,
)
text_repr = text_entry.to_api_repr()
print(f"TextEntry repr: {text_repr}")
assert text_repr["textPayload"] == "hello from the smoke test", text_repr
assert text_repr["severity"] == "WARNING", text_repr
assert text_repr["logName"] == "projects/demo-project/logs/demo", text_repr
assert text_repr["resource"] == {"type": "global", "labels": {"project_id": "demo-project"}}, text_repr

struct_entry = StructEntry(log_name="projects/demo-project/logs/demo", payload={"user": "alice", "count": 3})
struct_repr = struct_entry.to_api_repr()
print(f"StructEntry repr: {struct_repr}")
assert struct_repr["jsonPayload"] == {"user": "alice", "count": 3}, struct_repr

## Example 3: Structured JSON logging handler

In [ ]:
# Example 3: StructuredLogHandler writes standard-library log records as one JSON object per line,
# in the format Cloud Logging agents parse from stdout.
import io
import json
import logging

from google.cloud.logging.handlers import StructuredLogHandler

stream = io.StringIO()
handler = StructuredLogHandler(stream=stream, project_id="demo-project", labels={"env": "test"})
demo_logger = logging.getLogger("smoke_test.structured")
demo_logger.propagate = False
demo_logger.setLevel(logging.INFO)
demo_logger.addHandler(handler)
try:
    demo_logger.warning("disk usage at %d%%", 91)
    demo_logger.info({"event": "login", "user": "alice"})
finally:
    demo_logger.removeHandler(handler)

lines = stream.getvalue().strip().splitlines()
print(f"lines written: {len(lines)}")
# The first record a process writes is preceded by one diagnostic line naming the library version; skip it.
records = [r for r in map(json.loads, lines) if "logging.googleapis.com/diagnostic" not in r]
for record in records:
    print(f"  severity={record['severity']} payload={ {k: v for k, v in record.items() if not k.startswith(('logging.googleapis.com/', 'severity', 'httpRequest'))} }")
assert len(records) == 2, records
assert records[0]["severity"] == "WARNING", records[0]
assert records[0]["message"] == "disk usage at 91%", records[0]
assert records[0]["logging.googleapis.com/labels"]["env"] == "test", records[0]
assert "logging.googleapis.com/sourceLocation" in records[0], records[0]
assert records[1]["severity"] == "INFO" and records[1]["event"] == "login", records[1]

## Example 4: Batching entries locally

In [ ]:
# Example 4: A Batch collects entries locally and sends them in one API call on commit().
# Nothing is committed here, so no request is made.
import google.cloud.logging
from google.auth.credentials import AnonymousCredentials
from google.cloud.logging import StructEntry, TextEntry

client = google.cloud.logging.Client(project="demo-project", credentials=AnonymousCredentials())
logger = client.logger("demo-batch")
batch = logger.batch()
batch.log_text("first message", severity="INFO")
batch.log_struct({"step": 2, "status": "ok"}, severity="NOTICE")
batch.log_text("third message", labels={"attempt": "1"})

print(f"entries queued: {len(batch.entries)}")
for entry in batch.entries:
    print(f"  {type(entry).__name__}: severity={entry.severity} payload={entry.payload}")
assert [type(e) for e in batch.entries] == [TextEntry, StructEntry, TextEntry], batch.entries
assert batch.entries[1].payload == {"step": 2, "status": "ok"}, batch.entries[1]
assert batch.entries[2].labels == {"attempt": "1"}, batch.entries[2]

## Example 5: Sinks and metrics

In [ ]:
# Example 5: Sink and Metric resources: names and request bodies are built locally.
import google.cloud.logging
from google.auth.credentials import AnonymousCredentials

client = google.cloud.logging.Client(project="demo-project", credentials=AnonymousCredentials())

sink = client.sink(
    "demo-sink",
    filter_="severity>=ERROR",
    destination="storage.googleapis.com/demo-bucket",
)
print(f"sink full_name: {sink.full_name}")
print(f"sink destination: {sink.destination}")
assert sink.full_name == "projects/demo-project/sinks/demo-sink", sink.full_name
assert sink.path == "/projects/demo-project/sinks/demo-sink", sink.path
assert (sink.filter_, sink.destination) == ("severity>=ERROR", "storage.googleapis.com/demo-bucket")

metric = client.metric("error-count", filter_="severity>=ERROR", description="Errors per minute")
print(f"metric full_name: {metric.full_name}")
assert metric.full_name == "projects/demo-project/metrics/error-count", metric.full_name
assert metric.path == "/projects/demo-project/metrics/error-count", metric.path
assert (metric.filter_, metric.description) == ("severity>=ERROR", "Errors per minute")